# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All data entities—record sets, fields, columns—are referenced by their `@id` for consistency and reproducibility.

### Dataset Source

The dataset schema is specified by a Croissant JSON-LD available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant pandas matplotlib

## 1. Data Loading
Load the dataset metadata and initialize the dataset object with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print summary information from metadata
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else 'N/A'}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")
print(f"Available Record Sets: {[rs['@id'] for rs in getattr(metadata, 'recordSet', [])]}")

## 2. Data Overview

Review and inspect the available record sets and the structure (fields, columns, their `@id`), according to the Croissant schema.

Since record sets, fields, and columns are referenced by their `@id`, let's enumerate those in this dataset.

In [ ]:
# Get all record set @ids
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if isinstance(rs, dict):
            record_sets.append(rs['@id'])
        elif hasattr(rs, '@id'):
            record_sets.append(rs['@id'])

if not record_sets:
    # As of the metadata, recordSet is empty. Load from the dataset object as fallback if available.
    # `dataset.record_sets` will list registered record sets
    record_sets = list(dataset.record_sets.keys())

print("Record Sets @id:")
for rsid in record_sets:
    print(f"  - {rsid}")

# For each record set, get fields and their @ids
for rsid in record_sets:
    print(f"\nRecord Set: {rsid}")
    rs_schema = dataset.record_sets[rsid]
    fields = rs_schema.fields
    print("Fields and columns by @id:")
    for f in fields:
        field_id = f.get('@id', None)
        name = f.get('name', '-') if isinstance(f, dict) else getattr(f, 'name', '-')
        field_type = f.get('dataType', '-') if isinstance(f, dict) else getattr(f, 'dataType', '-')
        print(f"    Field name: {name}, @id: {field_id}, dataType: {field_type}")

## 3. Data Extraction
Let's load data from all record sets into Pandas DataFrames. All further references to data will use record set and field `@id`s. You should update `record_sets` as needed (see output above).

In [ ]:
dataframes = {}

if not record_sets:
    print("No record sets found in the metadata.")
else:
    for record_set_id in record_sets:
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:
                dataframes[record_set_id] = pd.DataFrame(records)
                print(f"Loaded {len(dataframes[record_set_id])} records for record set '{record_set_id}'.")
            else:
                print(f"No records found for record set '{record_set_id}'.")
        except Exception as e:
            print(f"Failed to load record set '{record_set_id}': {e}")

# Show columns for the first (if any) record set:
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nAvailable fields (columns) in '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

We will now perform data processing on the first available record set, demonstrating filtering, normalization, and grouping using field `@id`s only. Please adjust field IDs according to your own field listing above.

In [ ]:
# Pick a record set to analyze
if not dataframes:
    print("No dataframes loaded; EDA cannot run.")
else:
    record_set_id = list(dataframes.keys())[0]  # Use the first loaded record set
    df = dataframes[record_set_id]
    print(f"Exploring record set: {record_set_id}\nColumns: {list(df.columns)}")

    # Attempt to automatically select a numeric field for demonstration
    import numpy as np
    numeric_field = None
    for col in df.columns:
        # Try to detect a numeric column by data type or sample values
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
        # Try casting sample to float
        try:
            sample_val = df[col].dropna().iloc[0]
            float(sample_val)
            numeric_field = col
            break
        except Exception:
            continue
    if not numeric_field:
        print("No obvious numeric field detected. Please adjust to your dataset.")
    else:
        print(f"Using numeric field: {numeric_field}")
        # Set a basic threshold (mean if possible, or 0 as default)
        try:
            threshold = float(df[numeric_field].mean()) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        except Exception:
            threshold = 0

        # Ensure numeric conversion
        col_series = pd.to_numeric(df[numeric_field], errors='coerce')
        filtered_df = df[col_series > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (col_series - col_series.mean()) / col_series.std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another available field (try to select a likely categorical field by name)
        group_field = None
        candidates = [c for c in df.columns if c != numeric_field]
        for c in candidates:
            # Pick a field with low nunique that's not numeric
            nunique = df[c].nunique(dropna=True)
            if nunique > 1 and nunique < min(10, len(df)), not pd.api.types.is_numeric_dtype(df[c]):
                group_field = c
                break
        if group_field:
            print(f"\nGrouping by field: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped.head())
        else:
            print("No suitable non-numeric group field detected.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field for illustrative EDA.

In [ ]:
if not dataframes or not numeric_field:
    print("No loaded DataFrame or numeric field for visualization.")
else:
    plt.figure(figsize=(8,4))
    col_series = pd.to_numeric(df[numeric_field], errors='coerce')
    col_series = col_series.dropna()
    plt.hist(col_series, bins=20, color='skyblue', edgecolor='black', alpha=0.8)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.grid(axis='y')
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library. We programmatically examined available record sets and fields by their `@id`, loaded data using `mlcroissant.Dataset.records()`, and performed and visualized simple exploratory analysis using only `@id` field references.

For more details or to use specific fields/record sets, refer to the outputs in step 2. Further statistical analysis or domain-driven exploration is encouraged for research-specific tasks. See [FAIR² package](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) for metadata and documentation.